# Notebook 02 — Silver → Gold

## Objetivo
Leer las tablas depuradas de Silver y materializar en el Lakehouse Gold dos tablas Delta:

- `FactSales`: hechos de ventas.
- `DimCustomer`: dimensión de clientes.

Las tablas de origen están en `LH_Silver` y se llaman `Sales_Bronze` y `Customers_Bronze`. Aunque sus nombres incluyan el sufijo `_Bronze`, en este proyecto contienen los datos ya depurados de la capa Silver.

Al final se realiza una consulta de comprobación mediante un `JOIN` entre ambas tablas.

## Requisitos
- Lakehouse predeterminado de este notebook: **`LH_Silver`** (el mismo que el notebook 01, para poder compartir la sesión Spark en el pipeline).
- Existe un Lakehouse `LH_Gold` en el mismo workspace. No hace falta adjuntarlo: la escritura se hace por ruta ABFSS, que se obtiene automáticamente (sin identificadores en el código).
- Misma etiqueta de sesión en las actividades del pipeline y sin `%%configure` distintos entre notebooks.

---

## 1. Configuración

Se lee de `LH_Silver` (predeterminado) por nombre de tabla. Para escribir en `LH_Gold` se obtiene su ruta ABFSS de forma dinámica.


In [ ]:
from pyspark.sql.functions import col

# Ruta ABFSS de LH_Gold, obtenida del propio workspace (no se publican identificadores).
try:
    gold_root = notebookutils.lakehouse.get("LH_Gold")["properties"]["abfsPath"]
except Exception as e:
    raise RuntimeError(
        "No se pudo obtener la ruta de LH_Gold. Comprueba el nombre del Lakehouse "
        "o define gold_root manualmente con su ruta ABFSS."
    ) from e

# Si LH_Gold tiene esquemas habilitados, cambia a: gold_root + "/Tables/dbo"
gold_tables_path = gold_root + "/Tables/dbo"

print("Destino Gold:", gold_tables_path)

## 2. Leer las tablas Silver

Se leen `Sales_Bronze` y `Customers_Bronze` del lakehouse predeterminado (`LH_Silver`).

In [ ]:
sales_silver_df = spark.read.table("Sales_Bronze")
customers_silver_df = spark.read.table("Customers_Bronze")

## 3. Comprobar los datos de entrada

In [ ]:
print("Registros de ventas Silver:", sales_silver_df.count())
print("Registros de clientes Silver:", customers_silver_df.count())

## 4. Crear la tabla de hechos `FactSales`

In [ ]:
fact_sales_df = sales_silver_df.select(
    "SaleId",
    "SaleDate",
    "CustomerId",
    "Product",
    "Quantity",
    "UnitPrice",
    "TotalAmount"
)

## 5. Guardar `FactSales` en Gold

Se escribe en `LH_Gold` por ruta ABFSS, de modo que el resultado no depende del lakehouse predeterminado.

> **Importante:** `overwrite` reemplaza el contenido existente de la tabla. Es adecuado para este laboratorio de carga completa, pero debe evaluarse antes de utilizarlo en un proceso productivo.

In [ ]:
(
    fact_sales_df.write
    .format("delta")
    .mode("overwrite")
    .save(f"{gold_tables_path}/FactSales")
)

## 6. Crear la dimensión `DimCustomer`

In [ ]:
dim_customer_df = customers_silver_df.select(
    "CustomerId",
    "CustomerName",
    "City",
    "Segment"
)

## 7. Guardar `DimCustomer` en Gold

In [ ]:
(
    dim_customer_df.write
    .format("delta")
    .mode("overwrite")
    .save(f"{gold_tables_path}/DimCustomer")
)

## 8. Comprobar las tablas Gold

Se leen desde `LH_Gold` por ruta (no con `spark.table`, que apuntaría a `LH_Silver`).

In [ ]:
fact_gold_check = spark.read.format("delta").load(f"{gold_tables_path}/FactSales")
dim_gold_check = spark.read.format("delta").load(f"{gold_tables_path}/DimCustomer")

print("FactSales en Gold:", fact_gold_check.count())
print("DimCustomer en Gold:", dim_gold_check.count())

display(fact_gold_check)
display(dim_gold_check)

## 9. Probar la relación entre hechos y dimensión

Se realiza un `LEFT JOIN` utilizando `CustomerId` para enriquecer las ventas con los atributos del cliente. La relación formal `1:*` se configura después en el modelo semántico de Power BI.

In [ ]:
sales_analysis_df = (
    fact_gold_check.alias("f")
    .join(
        dim_gold_check.alias("d"),
        col("f.CustomerId") == col("d.CustomerId"),
        "left"
    )
    .select(
        col("f.SaleId"),
        col("f.SaleDate"),
        col("d.CustomerName"),
        col("d.City"),
        col("d.Segment"),
        col("f.Product"),
        col("f.TotalAmount")
    )
)

display(sales_analysis_df)

## Resultado esperado

Al finalizar, `LH_Gold` debe contener `FactSales` y `DimCustomer` (si no aparecen en el explorador, pulsa *Refresh* en el lakehouse). El `JOIN` final muestra las ventas enriquecidas con los datos del cliente.